# MSD publication figure from deposited summaries

The two publication cells are copied verbatim from the manuscript notebook. The setup cell only copies deposited summary tables into a writable output directory. Raw trajectory processing is not repeated in this notebook.


In [ ]:
from pathlib import Path
import os
import shutil

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "analyses").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the Gao2026 code root")

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
GAO2026_DATA_ROOT = Path(
    os.environ.get("GAO2026_DATA_ROOT", REPO_ROOT / "data")
).expanduser().resolve()
GAO2026_OUTPUT_ROOT = Path(
    os.environ.get("GAO2026_OUTPUT_ROOT", REPO_ROOT / "outputs")
).expanduser().resolve()

input_dir = GAO2026_DATA_ROOT / "07_MSD"
work_dir = GAO2026_OUTPUT_ROOT / "07_MSD"
work_dir.mkdir(parents=True, exist_ok=True)
for filename in ("replicate_msd_summary.csv", "replicate_anomalous_fit.csv"):
    shutil.copyfile(input_dir / filename, work_dir / filename)
os.environ["MSD_ALL_REPLICATES_OUTPUT_DIR"] = str(work_dir)


In [ ]:
# Cell 1: load all required data and packages without rerunning the analysis
from pathlib import Path
import os

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from IPython.display import display

OUTPUT_DIR = Path(
    os.environ.get(
        "MSD_ALL_REPLICATES_OUTPUT_DIR",
        str(GAO2026_OUTPUT_ROOT / '07_MSD'),
    )
)

MSD_SUMMARY_PATH = OUTPUT_DIR / "replicate_msd_summary.csv"
MSD_FIT_PATH = OUTPUT_DIR / "replicate_anomalous_fit.csv"

for required_path in [MSD_SUMMARY_PATH, MSD_FIT_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

msd_summary = pd.read_csv(MSD_SUMMARY_PATH)
msd_fits = pd.read_csv(MSD_FIT_PATH)

required_summary_columns = {
    "gene",
    "replicate",
    "state",
    "lag_frame",
    "lag_time_s",
    "n_trajectories",
    "mean_msd_um2",
    "ci95_low_um2",
    "ci95_high_um2",
}
required_fit_columns = {
    "gene",
    "replicate",
    "state",
    "fit_success",
    "D_alpha",
    "alpha",
}

missing_summary_columns = required_summary_columns - set(msd_summary.columns)
missing_fit_columns = required_fit_columns - set(msd_fits.columns)

if missing_summary_columns:
    raise ValueError(
        f"Missing columns in {MSD_SUMMARY_PATH.name}: "
        f"{sorted(missing_summary_columns)}"
    )
if missing_fit_columns:
    raise ValueError(
        f"Missing columns in {MSD_FIT_PATH.name}: "
        f"{sorted(missing_fit_columns)}"
    )

PLOT_GENES = ["Usp5", "Dnmt3L"]
PLOT_REPLICATES = [1, 2]
PANEL_LABELS = ["a", "b", "c", "d"]

STATE_ORDER = ["ON", "OFF"]
STATE_LABELS = {
    "ON": "Active",
    "OFF": "Inactive",
}
STATE_COLORS = {
    "ON": "#D55E00",
    "OFF": "#0072B2",
}

FONT_NAME = "Arial"

# Register Arial explicitly when it is available at a common macOS/Linux path.
arial_font_paths = [
    Path("/System/Library/Fonts/Supplemental/Arial.ttf"),
    Path("/System/Library/Fonts/Supplemental/Arial Bold.ttf"),
    Path("/System/Library/Fonts/Supplemental/Arial Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Arial Bold Italic.ttf"),
    Path("/usr/share/fonts/truetype/msttcorefonts/Arial.ttf"),
    Path("/usr/share/fonts/truetype/msttcorefonts/Arial_Bold.ttf"),
    Path("/usr/share/fonts/truetype/msttcorefonts/Arial_Italic.ttf"),
    Path("/usr/share/fonts/truetype/msttcorefonts/Arial_Bold_Italic.ttf"),
    Path.home() / ".fonts/Arial.ttf",
]

for font_path in arial_font_paths:
    if font_path.exists():
        font_manager.fontManager.addfont(str(font_path))

available_font_names = {
    font_entry.name for font_entry in font_manager.fontManager.ttflist
}
if FONT_NAME not in available_font_names:
    raise RuntimeError(
        "Arial is not available in this Python environment. "
        "Install or register Arial before generating the final PDF."
    )

OUTPUT_PDF = (
    OUTPUT_DIR
    / "MSD_Active_Inactive_by_biological_replicate_"
    "no_Nanog_publication_Arial.pdf"
)

print(f"MSD summary: {MSD_SUMMARY_PATH}")
print(f"Fit results: {MSD_FIT_PATH}")
print(f"Output PDF: {OUTPUT_PDF}")
print(f"Font: {FONT_NAME}")

In [ ]:
# Cell 2: generate and export the publication-style PDF
def fit_succeeded(value) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() == "true"


FONT_size = 10
PANEL_LABELS = ["d", "e", "f", "g"]

def draw_msd_panel(
    axis: plt.Axes,
    gene: str,
    replicate: int,
) -> None:
    panel_summary = msd_summary[
        (msd_summary["gene"] == gene)
        & (msd_summary["replicate"] == replicate)
    ].copy()

    panel_fits = msd_fits[
        (msd_fits["gene"] == gene)
        & (msd_fits["replicate"] == replicate)
    ].copy()

    if panel_summary.empty:
        raise ValueError(
            f"No MSD summary was found for {gene}, replicate {replicate}."
        )

    for state in STATE_ORDER:
        state_summary = panel_summary[
            panel_summary["state"] == state
        ].sort_values("lag_frame")

        if state_summary.empty:
            continue

        x = state_summary["lag_time_s"].to_numpy(dtype=float)
        mean_msd = state_summary["mean_msd_um2"].to_numpy(dtype=float)
        ci_low = np.maximum(
            0.0,
            state_summary["ci95_low_um2"].to_numpy(dtype=float),
        )
        ci_high = state_summary["ci95_high_um2"].to_numpy(dtype=float)
        n_trajectories = int(state_summary["n_trajectories"].max())

        axis.fill_between(
            x,
            ci_low,
            ci_high,
            color=STATE_COLORS[state],
            alpha=0.18,
            linewidth=0,
        )
        axis.plot(
            x,
            mean_msd,
            color=STATE_COLORS[state],
            linewidth=1.5,
            label=f"{STATE_LABELS[state]} (n = {n_trajectories})",
        )

        fit_row = panel_fits[panel_fits["state"] == state]
        if (
            not fit_row.empty
            and fit_succeeded(fit_row.iloc[0]["fit_success"])
        ):
            fit_row = fit_row.iloc[0]
            x_fit = np.linspace(
                max(0.001, float(x.min())),
                float(x.max()),
                300,
            )
            y_fit = (
                4.0
                * float(fit_row["D_alpha"])
                * np.power(x_fit, float(fit_row["alpha"]))
            )

            axis.plot(
                x_fit,
                y_fit,
                color=STATE_COLORS[state],
                linewidth=0.9,
                linestyle=(0, (3, 2)),
            )

    # Separate text objects keep only the gene name italicized.
    axis.text(
        0.5,
        1.075,
        gene,
        transform=axis.transAxes,
        ha="center",
        va="bottom",
        fontsize=FONT_size,
        fontfamily=FONT_NAME,
        fontstyle="italic",
    )
    axis.text(
        0.5,
        1.015,
        f"Replicate {replicate}",
        transform=axis.transAxes,
        ha="center",
        va="bottom",
        fontsize=FONT_size,
        fontfamily=FONT_NAME,
    )

    axis.set_xlim(0.0, 2.0)
    axis.set_ylim(0.0, 0.04)
    axis.set_xticks(np.arange(0.0, 2.01, 0.5))
    axis.set_yticks(np.arange(0.0, 0.0401, 0.01))

    axis.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=3,
        width=0.7,
        pad=2,
        labelsize=FONT_size,
    )

    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.spines["left"].set_linewidth(0.8)
    axis.spines["bottom"].set_linewidth(0.8)

    axis.legend(
        loc="upper left",
        frameon=False,
        fontsize=FONT_size,
        handlelength=2.4,
        borderaxespad=0.2,
        labelspacing=0.25,
    )



publication_rc = {
    "font.family": FONT_NAME,
    "font.sans-serif": [FONT_NAME],
    "font.size": FONT_size,
    "axes.labelsize": FONT_size,
    "axes.titlesize": FONT_size,
    "axes.linewidth": 0.8,
    "xtick.labelsize": FONT_size,
    "ytick.labelsize": FONT_size,
    "legend.fontsize": FONT_size,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}

with mpl.rc_context(publication_rc):
    figure, axes = plt.subplots(
        2,
        2,
        figsize=(180 / 25.4, 142 / 25.4),
        sharex=True,
        sharey=True,
    )

    panel_index = 0
    for row_index, replicate in enumerate(PLOT_REPLICATES):
        for column_index, gene in enumerate(PLOT_GENES):
            axis = axes[row_index, column_index]

            draw_msd_panel(
                axis=axis,
                gene=gene,
                replicate=replicate,
            )

            axis.text(
                -0.17,
                1.09,
                PANEL_LABELS[panel_index],
                transform=axis.transAxes,
                ha="left",
                va="bottom",
                fontsize=FONT_size * 1.33,
                fontfamily=FONT_NAME,
                fontweight="bold",
            )
            panel_index += 1

    for axis in axes[-1, :]:
        axis.set_xlabel("Time lag (s)")

    for axis in axes[:, 0]:
        axis.set_ylabel(r"MSD ($\mathrm{\mu m}^2$)")

    figure.subplots_adjust(
        left=0.105,
        right=0.985,
        bottom=0.105,
        top=0.925,
        wspace=0.20,
        hspace=0.32,
    )

    figure.savefig(
        OUTPUT_PDF,
        format="pdf",
        bbox_inches="tight",
        pad_inches=0.03,
        facecolor="white",
    )

    display(figure)
    plt.close(figure)

if not OUTPUT_PDF.exists() or OUTPUT_PDF.stat().st_size == 0:
    raise RuntimeError(f"PDF export failed: {OUTPUT_PDF}")

print(f"Saved: {OUTPUT_PDF}")